# Custom Gymnasium Env: Chrome Dino

- Chrome browser window w/ `chrome://dino`
- OBS Studio pipewire record that window
- Plug & Play remote code to control
- Wrap all in a Gymnasium env


## Capture window

In [15]:
!uv add pipewire-capture PyGObject opencv-python

Resolved 163 packages in 0.76ms
Audited 159 packages in 0.64ms


In [14]:
import cv2
import numpy as np
import gi
from pipewire_capture import PortalCapture, CaptureStream, is_available

# Initialize GStreamer
gi.require_version('Gst', '1.0')
from gi.repository import Gst, GLib

Gst.init(None)

class ScreenCaptureViewer:
    def __init__(self, fd, node_id, width, height):
        self.frame = None
        self.fd = fd
        self.node_id = node_id
        self.width = width
        self.height = height
        
        # Build GStreamer pipeline to read from PipeWire
        # pipewiresrc: Connects to the granted PipeWire stream
        # fd: The file descriptor from the portal session
        # videoconvert + format=BGR: Convert to OpenCV-compatible format
        pipeline_str = (
            f"pipewiresrc fd={fd} path={node_id} ! "
            "video/x-raw ! "
            "videoconvert ! "
            "video/x-raw, format=BGR ! "
            "appsink name=sink emit-signals=true sync=false"
        )
        
        self.pipeline = Gst.parse_launch(pipeline_str)
        self.sink = self.pipeline.get_by_name("sink")
        self.sink.connect("new-sample", self.on_new_sample)
        
    def on_new_sample(self, sink):
        """Callback when a new frame is ready"""
        sample = sink.emit("pull-sample")
        if sample:
            buffer = sample.get_buffer()
            caps = sample.get_caps()
            
            # Get dimensions from caps
            structure = caps.get_structure(0)
            width = structure.get_value("width")
            height = structure.get_value("height")
            
            # Map buffer to numpy array
            success, map_info = buffer.map(Gst.MapFlags.READ)
            if success:
                self.frame = np.ndarray(
                    (height, width, 3),
                    buffer=map_info.data,
                    dtype=np.uint8
                ).copy()  # .copy() is important!
                buffer.unmap(map_info)
            
            return Gst.FlowReturn.OK
        return Gst.FlowReturn.ERROR
    
    def start(self):
        self.pipeline.set_state(Gst.State.PLAYING)
    
    def stop(self):
        self.pipeline.set_state(Gst.State.NULL)
    
    def get_frame(self):
        return self.frame

def main():
    # Check if PipeWire capture is available (Wayland + Portal)
    if not is_available():
        print("❌ PipeWire capture not available")
        print("   Make sure you're running on Wayland with xdg-desktop-portal")
        exit(1)
    
    print("✅ PipeWire capture available")
    print("   Opening system screen picker...")
    
    # Trigger the system popup (same as OBS!)
    portal = PortalCapture()
    session = portal.select_window()  # or select_screen() depending on API
    
    if not session:
        print("❌ No selection made or permission denied")
        exit(1)
    
    print(f"✅ Session granted!")
    print(f"   Node ID: {session.node_id}")
    print(f"   Resolution: {session.width}x{session.height}")
    
    # Create the stream viewer
    viewer = ScreenCaptureViewer(
        fd=session.fd,
        node_id=session.node_id,
        width=session.width,
        height=session.height
    )
    
    viewer.start()
    print("🎬 Capture started. Press 'q' to quit.")
    
    # Main display loop
    try:
        while True:
            frame = viewer.get_frame()
            
            if frame is not None:
                # Optional: Add FPS counter
                cv2.putText(frame, "Press 'q' to quit", (10, 30),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                
                cv2.imshow("PipeWire Screen Capture", frame)
            
            # Wait 1ms for key press
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
    
    except KeyboardInterrupt:
        print("\n⚠ Interrupted by user")
    
    finally:
        viewer.stop()
        session.close()
        cv2.destroyAllWindows()
        print("✅ Cleanup complete")

if __name__ == "__main__":
    main()

✅ PipeWire capture available
   Opening system screen picker...
✅ Session granted!
   Node ID: 122
   Resolution: 0x0
🎬 Capture started. Press 'q' to quit.
✅ Cleanup complete


---

## Websocket based exec

In [17]:
!uv add websockets

Resolved 164 packages in 883ms                                       
Prepared 1 package in 339ms                                              
Installed 1 package in 14ms                                 
 + websockets==16.0


### Client-Side: data send

run it in console
```js
(function () {
    const WS_SERVER = 'ws://localhost:8766';
    let ws = null;
    let reconnectTimeout = null;
    const PING_INTERVAL = 5_000; // 10 seconds
    let pingInterval = null;

    function connect() {
        ws = new WebSocket(WS_SERVER);

        ws.onopen = () => {
            console.log('[RemoteExec] Connected to WebSocket server');
            startPing();
        };

        ws.onmessage = async (event) => {
            try {
                const data = JSON.parse(event.data);

                if (data.type === 'pong') {
                    // Keep-alive response, nothing to do
                    return;
                }

                if (data.type === 'execute') {
                    const code = data.code;

                    // Wrap user code in an async function
                    let result;
                    try {
                        // Use new Function to avoid eval restrictions and enable async
                        const fn = new Function('return (async () => {' + code + '})()');
                        result = fn(); // This returns a Promise
                    } catch (e) {
                        // Syntax error or function creation failed
                        sendResult({
                            type: 'execution_result',
                            uid: data.uid,
                            result: null,
                            error: '[JS Parse Error] ' + (e.stack || e.toString()),
                        });
                        return;
                    }

                    // Handle Promise resolution
                    Promise.resolve(result)
                        .then(value => {
                            // Serialize result safely
                            try {
                                sendResult({
                                    type: 'execution_result',
                                    uid: data.uid,
                                    result: value,
                                    error: null,
                                });
                            } catch (e) {
                                sendResult({
                                    type: 'execution_result',
                                    uid: data.uid,
                                    result: null,
                                    error: '[Serialize Error] ' + (e.stack || e.toString()),
                                });
                            }
                        })
                        .catch(err => {
                            sendResult({
                                type: 'execution_result',
                                uid: data.uid,
                                error: String(err),
                                result: null,
                            });
                        });
                }
            } catch (e) {
                console.error('[RemoteExec] Error processing message:', e);
            }
        };

        ws.onclose = () => {
            console.log('[RemoteExec] Disconnected from WebSocket server');
            stopPing();
            scheduleReconnect();
        };

        ws.onerror = (error) => {
            console.error('[RemoteExec] WebSocket error:', error);
        };
    }

    function sendResult(payload) {
        if (ws && ws.readyState === WebSocket.OPEN) {
            ws.send(JSON.stringify(payload));
        }
    }

    function startPing() {
        stopPing();
        pingInterval = setInterval(() => {
            if (ws && ws.readyState === WebSocket.OPEN) {
                ws.send(JSON.stringify({
                    type: 'ping',
                    uid: crypto.randomUUID(),
                    timestamp: Date.now()
                }));
            }
        }, PING_INTERVAL);
    }

    function stopPing() {
        if (pingInterval) {
            clearInterval(pingInterval);
            pingInterval = null;
        }
    }

    function scheduleReconnect() {
        if (reconnectTimeout) {
            clearTimeout(reconnectTimeout);
        }
        reconnectTimeout = setTimeout(() => {
            console.log('[RemoteExec] Reconnecting...');
            connect();
        }, 3000);
    }

    // Start connection
    connect();
})();
```

###  Server

In [1]:
import sys
from uuid import uuid5

import websockets
import json
import time
import threading
from collections import defaultdict
import asyncio
import uuid


class WebSocketServer:
    def __init__(self, host="localhost", port=8766):
        self.host = host
        self.port = port
        self.clients = {}  # {uid: websocket}
        self.client_uid_counter = 0
        self.client_last_seen = {}  # {uid: timestamp}
        self.TIMEOUT = 45  # seconds
        self.msg_callbacks = {}  # {msg_id: callback}
        self.loop = None

    async def handle_client(self, websocket):
        # Assign unique ID
        self.client_uid_counter += 1
        client_uid = self.client_uid_counter
        self.clients[client_uid] = websocket
        self.client_last_seen[client_uid] = time.time()
        print(f"Client {client_uid} connected", flush=True, file=sys.stdout)

        try:
            async for message in websocket:
                try:
                    print(f"From {client_uid}: {message}", flush=True)
                    data = json.loads(message)

                    # validation
                    """
                    {
                      type: "ping" | "pong" | "execute" | "execution_result"
                      uid: str | int
                      timestamp: number (required only for 'ping'/'pong' messages)
                      result: any (required only for 'execution_result' messages | if no error)
                      error: any (required only for 'execution_result' messages | only if error occurs)
                      code: str (required only for 'execute' messages)
                    }
                    """
                    assert isinstance(data, dict), "Data must be a dictionary"
                    assert 'type' in data, "Data must contain 'type' field"
                    assert data['type'] in ['ping', 'pong', 'execute',
                                            'execution_result'], f"Invalid type: {data['type']}"
                    assert 'uid' in data, "Data must contain 'uid' field"
                    assert isinstance(data['uid'], (str, int)), "UID must be string or int"
                    if data['type'] in ['ping', 'pong']:
                        assert 'timestamp' in data, f"Timestamp required for {data['type']}"
                        assert isinstance(data['timestamp'], (int, float)), "Timestamp must be numeric"
                    if data['type'] == 'execute':
                        assert 'code' in data, "Code required for execute"
                        assert isinstance(data['code'], str), "Code must be string"
                    if data['type'] == 'execution_result':
                        assert 'result' in data, "Result required for execution_result"
                        assert 'error' in data, "Error required for execution_result"

                    await self.handle_message(data, client_uid)
                except json.JSONDecodeError:
                    print(f"Invalid JSON from client {client_uid}")
                except AssertionError as e:
                    print(f"Warning: Invalid message format from client {client_uid}: {e}")
        except websockets.exceptions.ConnectionClosed:
            print(f"Client {client_uid} disconnected")
        finally:
            if client_uid in self.clients:
                del self.clients[client_uid]
                if client_uid in self.client_last_seen:
                    del self.client_last_seen[client_uid]

    async def handle_message(self, data, client_uid):
        msg_type = data.get('type')
        msg_id = data.get('uid')

        # Update last seen time
        self.client_last_seen[client_uid] = time.time()

        if msg_type == 'ping':
            await self.send_message(data={'uid': msg_id, 'type': 'pong', 'timestamp': time.time()},
                                    client_uid=client_uid)

        elif msg_type == 'execution_result':
            if msg_id in self.msg_callbacks:
                callback = self.msg_callbacks.pop(msg_id)
                asyncio.create_task(callback(data))

        else:
            print("Warn: msg has no callback!")

    async def send_message(self, data, callback=None, client_uid=None):
        client_uids = [client_uid] if client_uid is not None else self.clients.keys()

        for client_uid in client_uids:
            if client_uid not in self.clients:
                continue

            try:
                # Add message ID if not present
                if 'uid' not in data:
                    data['uid'] = str(uuid.uuid4())

                # Register callback if provided
                if callback is not None:
                    self.msg_callbacks[data['uid']] = callback

                await self.clients[client_uid].send(json.dumps(data))
                print(f"Sent message to client {client_uid}: {data}")  # Log sent message
            except websockets.exceptions.ConnectionClosed:
                # Clean up dead connection
                del self.clients[client_uid]
                if client_uid in self.client_last_seen:
                    del self.client_last_seen[client_uid]

    async def __cleanup_dead_connections(self):
        while True:
            current_time = time.time()
            dead_clients = [
                client_uid for client_uid, last_seen in self.client_last_seen.items()
                if current_time - last_seen > self.TIMEOUT
            ]

            for client_uid in dead_clients:
                print(f"Client {client_uid} timed out")
                if client_uid in self.clients:
                    await self.clients[client_uid].close()
                    del self.clients[client_uid]
                if client_uid in self.client_last_seen:
                    del self.client_last_seen[client_uid]

            await asyncio.sleep(5)  # Check every 5 seconds

    def start_server(self):
        async def run_server():
            server = await websockets.serve(self.handle_client, self.host, self.port)
            print(f"WebSocket server started on ws://{self.host}:{self.port}")

            # Start cleanup task
            cleanup_task = asyncio.create_task(self.__cleanup_dead_connections())

            await server.wait_closed()

        # Create a new event loop for this thread
        self.loop = asyncio.new_event_loop()
        asyncio.set_event_loop(self.loop)
        self.loop.run_until_complete(run_server())

    async def execute_js(self, code, timeout=0.01):
        # Generate a unique message ID for this execution
        msg_id = str(uuid.uuid4())

        # Create a future to wait for the response
        future = asyncio.Future()

        # Define the callback that will set the result of the future
        async def callback(data):
            if not future.done():
                future.set_result(data)

        # Send the execute message with the callback
        await self.send_message(
            data={
                "type": "execute",
                "code": code,
                "uid": msg_id
            },
            callback=callback
        )

        # Wait for the callback to be called and return the result
        result = await asyncio.wait_for(future, timeout=timeout)
        return result

In [2]:
server = WebSocketServer()

server_thread = threading.Thread(target=lambda: server.start_server(), daemon=True)
server_thread.start()

time.sleep(3)  # just wait, to let the client connect

WebSocket server started on ws://localhost:8766
Client 1 connected
From 1: {"type":"ping","uid":"c8bcd2cf-66ad-415e-83d5-06fce38e480f","timestamp":1772670135566}
Sent message to client 1: {'uid': 'c8bcd2cf-66ad-415e-83d5-06fce38e480f', 'type': 'pong', 'timestamp': 1772670135.5694098}


In [ ]:
await server.execute_js("return 2 + 58")

In [14]:
await server.execute_js("""
function simulateKeyPress(key, code) {
    const target = document.activeElement;
    
    if (!target) {
        console.warn("No element is currently focused.");
        return;
    }

    const commonProps = {
        key: key,
        code: code,
        bubbles: true,
        cancelable: true,
        view: window,
        which: key.charCodeAt(0), // Fallback for older listeners
        keyCode: key.charCodeAt(0) // Fallback for older listeners
    };

    // 1. Dispatch KeyDown
    const downEvent = new KeyboardEvent('keydown', commonProps);
    target.dispatchEvent(downEvent);

    // 2. Dispatch KeyPress (Legacy, but some games/apps still listen to this)
    // Note: 'keypress' is deprecated, but included for maximum compatibility
    if (key !== 'ArrowUp' && key !== 'ArrowDown') { 
        // Arrow keys typically don't fire 'keypress' in modern browsers, 
        // but we include it for the Spacebar just in case.
        const pressEvent = new KeyboardEvent('keypress', {
            ...commonProps,
            charCode: key.charCodeAt(0)
        });
        target.dispatchEvent(pressEvent);
    }

    // 3. Dispatch KeyUp
    const upEvent = new KeyboardEvent('keyup', commonProps);
    target.dispatchEvent(upEvent);
}

//simulateKeyPress('ArrowDown', 'ArrowDown')
//simulateKeyPress('ArrowUp', 'ArrowUp')
//simulateKeyPress(' ', 'Space')
return true;
""")

Sent message to client 1: {'type': 'execute', 'code': '\nfunction simulateKeyPress(key, code) {\n    const target = document.activeElement;\n\n    if (!target) {\n        console.warn("No element is currently focused.");\n        return;\n    }\n\n    const commonProps = {\n        key: key,\n        code: code,\n        bubbles: true,\n        cancelable: true,\n        view: window,\n        which: key.charCodeAt(0), // Fallback for older listeners\n        keyCode: key.charCodeAt(0) // Fallback for older listeners\n    };\n\n    // 1. Dispatch KeyDown\n    const downEvent = new KeyboardEvent(\'keydown\', commonProps);\n    target.dispatchEvent(downEvent);\n\n    // 2. Dispatch KeyPress (Legacy, but some games/apps still listen to this)\n    // Note: \'keypress\' is deprecated, but included for maximum compatibility\n    if (key !== \'ArrowUp\' && key !== \'ArrowDown\') { \n        // Arrow keys typically don\'t fire \'keypress\' in modern browsers, \n        // but we include it for

{'type': 'execution_result',
 'uid': '78ae09d1-f322-491c-9200-8d80c944d94f',
 'result': True,
 'error': None}

From 1: {"type":"ping","uid":"36d87219-38cc-4cfb-8b20-eb0e416374bc","timestamp":1772670455566}
Sent message to client 1: {'uid': '36d87219-38cc-4cfb-8b20-eb0e416374bc', 'type': 'pong', 'timestamp': 1772670455.5685165}
From 1: {"type":"ping","uid":"8001ba6a-252d-426d-927b-9d317617a8d8","timestamp":1772670460565}
Sent message to client 1: {'uid': '8001ba6a-252d-426d-927b-9d317617a8d8', 'type': 'pong', 'timestamp': 1772670460.5679555}
From 1: {"type":"ping","uid":"0998c9c4-6ded-4cfe-a051-5aa520db3d20","timestamp":1772670465566}
Sent message to client 1: {'uid': '0998c9c4-6ded-4cfe-a051-5aa520db3d20', 'type': 'pong', 'timestamp': 1772670465.5682514}
From 1: {"type":"ping","uid":"164946be-bdd9-4809-a07d-d41ecd55d7c6","timestamp":1772670470565}
Sent message to client 1: {'uid': '164946be-bdd9-4809-a07d-d41ecd55d7c6', 'type': 'pong', 'timestamp': 1772670470.5682032}
From 1: {"type":"ping","uid":"b5f1ddc2-db05-4c68-8e53-852030b45feb","timestamp":1772670475566}
Sent message to client 1: {'uid':

-----

# Chrome DevTools Protocol

In [15]:
!uv add pychrome

Resolved 166 packages in 1.01s                                       
Prepared 2 packages in 292ms                                             
Installed 2 packages in 14ms                                
 + click==8.3.1
 + pychrome==0.2.4


In [16]:
import asyncio
import json
import base64
import cv2
import numpy as np
import websockets

async def test_raw_cdp():
    # Get debugging endpoint
    async with websockets.connect("ws://localhost:9222/json/version") as ws:
        version_info = json.loads(await ws.recv())
        ws_url = version_info["webSocketDebuggerUrl"]
    
    async with websockets.connect(ws_url) as ws:
        # Helper to send command & wait for response
        async def send_cmd(method, **params):
            cmd_id = 1
            await ws.send(json.dumps({"id": cmd_id, "method": method, "params": params}))
            while True:
                msg = json.loads(await ws.recv())
                if msg.get("id") == cmd_id:
                    return msg
        
        # Enable domains
        await send_cmd("Page.enable")
        await send_cmd("Runtime.enable")
        
        # Navigate
        await send_cmd("Page.navigate", url="chrome://dino")
        await asyncio.sleep(2)
        
        # Screenshot
        result = await send_cmd("Page.captureScreenshot")
        img = cv2.imdecode(np.frombuffer(base64.b64decode(result['data']), dtype=np.uint8), cv2.IMREAD_COLOR)
        cv2.imwrite("dino_raw.png", img)
        print(f"✅ Raw CDP screenshot: {img.shape}")
        
        # Inject key
        for evt in ['keyDown', 'keyUp']:
            await send_cmd("Input.dispatchKeyEvent", type=evt, key=' ', code='Space', windowsVirtualKeyCode=32)
            await asyncio.sleep(0.05)
        
        print("🎉 Raw CDP test complete!")

# Run with: asyncio.run(test_raw_cdp())

In [ ]:
asyncio.run(test_raw_cdp())

In [19]:
# !chromium --remote-debugging-port=9222 \
#   --headless=new \
#   --window-size=800,200 \
#   --disable-gpu \
#   --no-sandbox \
#   chrome://dino

In [ ]:
#!/usr/bin/env python3
import base64
import cv2
import numpy as np
import time
import pychrome

In [26]:
def test_cdp_connection(port=9222):
    print(f"🔌 Connecting to Chrome CDP at localhost:{port}...")
    
    browser = pychrome.Browser(url=f"http://localhost:{port}")
    tab = browser.new_tab()
    
    try:
        tab.start()
        tab.Runtime.enable()
        tab.Page.enable()
        
        # Navigate to Dino (if not already there)
        print("🌐 Navigating to chrome://dino...")
        tab.Page.navigate(url="chrome://dino")
        time.sleep(2)  # Wait for load
        
        # 📸 Test 1: Capture screenshot
        print("📸 Capturing screenshot...")
        result = tab.Page.captureScreenshot()
        img_data = base64.b64decode(result['data'])
        frame = cv2.imdecode(np.frombuffer(img_data, dtype=np.uint8), cv2.IMREAD_COLOR)
        print(f"✅ Frame shape: {frame.shape}")
        cv2.imwrite("dino_capture.png", frame)
        
        # 🎮 Test 2: Inject Spacebar to start/jump
        print("⌨️  Injecting Spacebar (keyDown + keyUp)...")
        for event_type in ['keyDown', 'keyUp']:
            tab.Input.dispatchKeyEvent(
                type=event_type,
                key=' ',
                code='Space',
                windowsVirtualKeyCode=32,
                nativeVirtualKeyCode=32
            )
            time.sleep(0.05)
        
        # 📊 Test 3: Read game state via JS eval
        print("🔍 Reading game state...")
        score_result = tab.Runtime.evaluate(expression="parseInt(runnerInstance.distanceMeter.digits.join(''))")
        gameover_result = tab.Runtime.evaluate(expression="runnerInstance.crashed")
        
        score = score_result['result']['value']
        game_over = gameover_result['result']['value']
        print(f"📈 Score: {score} | 💀 Game Over: {game_over}")
        
        # 🔄 Test 4: Reset game (if needed)
        if game_over:
            print("🔄 Resetting game...")
            tab.Runtime.evaluate(expression="runnerInstance.restart()")
            time.sleep(0.5)
        
        print("\n🎉 All CDP tests passed! Ready for Gymnasium wrapper.")
        return True
        
    except Exception as e:
        print(f"❌ Test failed: {e}")
        import traceback
        traceback.print_exc()
        return False
    finally:
        tab.stop()
        # browser.close()

In [33]:
test_cdp_connection()

🔌 Connecting to Chrome CDP at localhost:9222...
🌐 Navigating to chrome://dino...
📸 Capturing screenshot...
✅ Frame shape: (513, 800, 3)
⌨️  Injecting Spacebar (keyDown + keyUp)...
🔍 Reading game state...
📈 Score: 0 | 💀 Game Over: False

🎉 All CDP tests passed! Ready for Gymnasium wrapper.


Exception in thread Thread-18 (_recv_loop):
Traceback (most recent call last):
  File "/home/ezio4df/.local/share/uv/python/cpython-3.13.5-linux-x86_64-gnu/lib/python3.13/threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/home/ezio4df/.local/share/uv/python/cpython-3.13.5-linux-x86_64-gnu/lib/python3.13/threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ezio4df/projects/play-TRexRunner/.venv/lib/python3.13/site-packages/pychrome/tab.py", line 122, in _recv_loop
    message = json.loads(message_json)
  File "/home/ezio4df/.local/share/uv/python/cpython-3.13.5-linux-x86_64-gnu/lib/python3.13/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ~~~~~~~~~~~~~~~~~~~~~~~^^^
  File "/home/ezio4df/.local/share/uv/python/cpython-3.13.5-linux-x86_64-gnu/lib/python3.13/json/decoder.py", line 345, in decode
    obj, end = self.raw_decode(s, idx=

True

🔌 Connecting to Chrome CDP at localhost:9222...
🌐 Navigating to chrome://dino...


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/ezio4df/projects/play-TRexRunner/.venv/lib/python3.13/site-packages/cv2/qt/plugins"


⌨️  Controls: ↑=Jump, ↓=Duck, Q=Quit
👋 Quit requested
✅ Clean exit


In [37]:
!uv add cdp-socket opencv-python numpy aiohttp

Resolved 175 packages in 49ms                                        
Audited 171 packages in 19ms


In [2]:
import asyncio, websockets, aiohttp, cv2, numpy as np, base64, json, threading, os

# Fix Qt/Wayland issues on Linux
os.environ["QT_QPA_PLATFORM"] = "xcb"

class DinoStream:
    def __init__(self):
        self.frame = None
        self.keys = asyncio.Queue()
        self.running = True
        self.ws = None
        self.msg_id = 0

    def cv2_thread(self):
        cv2.namedWindow("Dino", cv2.WINDOW_NORMAL)
        while self.running:
            if self.frame is not None:
                cv2.imshow("Dino", self.frame)
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'): 
                self.running = False
                break
            # 82/84 (Requested) + 65362/65364 (Standard Linux Arrows)
            if key in [82, 65362]:  # Up
                asyncio.run_coroutine_threadsafe(self.keys.put('UP'), self.loop)
            elif key in [84, 65364]:  # Down
                asyncio.run_coroutine_threadsafe(self.keys.put('DOWN'), self.loop)
        cv2.destroyAllWindows()

    async def get_ws_url(self, port):
        async with aiohttp.ClientSession() as sess:
            await sess.get(f"http://localhost:{port}/json/new?chrome://dino")
            async with sess.get(f"http://localhost:{port}/json/list") as resp:
                return (await resp.json())[0]['webSocketDebuggerUrl']

    async def send_key(self, action):
        code = 38 if action == 'UP' else 40
        name = "ArrowUp" if action == 'UP' else "ArrowDown"
        for t in ['keyDown', 'keyUp']:
            await self.ws.send(json.dumps({
                "id": 1, "method": "Input.dispatchKeyEvent",
                "params": {"type": t, "key": name, "code": name, "windowsVirtualKeyCode": code}
            }))

    async def cdp_loop(self, port):
        ws_url = await self.get_ws_url(port)
        async with websockets.connect(ws_url, ping_interval=None) as self.ws:
            await self.ws.send(json.dumps({"id": 1, "method": "Page.enable"}))
            await self.ws.recv()
            threading.Thread(target=self.cv2_thread, daemon=True).start()
            
            while self.running:
                self.msg_id += 1
                await self.ws.send(json.dumps({"id": self.msg_id, "method": "Page.captureScreenshot"}))
                resp = await self.ws.recv()
                data = json.loads(resp)
                
                # Fix KeyError: check structure
                if 'result' in data and 'data' in data['result']:
                    img = base64.b64decode(data['result']['data'])
                    self.frame = cv2.imdecode(np.frombuffer(img, np.uint8), cv2.IMREAD_COLOR)
                
                try:
                    action = self.keys.get_nowait()
                    await self.send_key(action)
                except asyncio.QueueEmpty: 
                    pass
                
                await asyncio.sleep(1/30)

    def run(self, port=9222):
        self.loop = asyncio.new_event_loop()
        asyncio.set_event_loop(self.loop)
        try:
            self.loop.run_until_complete(self.cdp_loop(port))
        except KeyboardInterrupt:
            pass
        finally:
            self.running = False
            self.loop.close()

if __name__ == "__main__":
    DinoStream().run()

In [3]:
DinoStream().run()

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/ezio4df/projects/play-TRexRunner/.venv/lib/python3.13/site-packages/cv2/qt/plugins"
Exception in thread Thread-5 (cv2_thread):
Traceback (most recent call last):
  File "/home/ezio4df/.local/share/uv/python/cpython-3.13.5-linux-x86_64-gnu/lib/python3.13/threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/home/ezio4df/.local/share/uv/python/cpython-3.13.5-linux-x86_64-gnu/lib/python3.13/threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_86210/1549755850.py", line 18, in cv2_thread
    loop = asyncio.get_running_loop()
RuntimeError: no running event loop


> the CDP aproach is undoubtly would be the best one, but after wasting 3h on it, i dont think i will take this approach
> TODOL figure otu how make CDP work LATER!